In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim
import logging
import matplotlib.pyplot as plt
from argparse import ArgumentParser
from torch.autograd import Variable
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.distributions import Normal, OneHotCategorical
import torch.nn.utils as nn_utils

torch.manual_seed(123456)
np.random.seed(123456)


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_normal_(m.weight)
        nn.init.constant_(m.bias, 0.0)        
class MDN(nn.Module):
    def __init__(self, n_hidden, n_gaussians,n_hidden_layers,bn, dout, dout_value):
        super(MDN, self).__init__()       
        layers = [nn.Linear(41, n_hidden), nn.Tanh()] #Input features
        if bn == 1:
            layers.append(nn.BatchNorm1d(n_hidden))    
        for _ in range(n_hidden_layers - 2):
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())
        if dout==1:
            layers.append(nn.Dropout(dout_value))
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())   
        else:
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())       
        self.z_h = nn.Sequential(*layers)             
        self.z_pi = nn.Linear(n_hidden, n_gaussians)
        self.z_sigma = nn.Linear(n_hidden, n_gaussians)
        self.z_mu = nn.Linear(n_hidden, n_gaussians)  
    def forward(self, x):
        z_h = self.z_h(x)
        pi = nn.functional.softmax(self.z_pi(z_h), -1)
        sigma = torch.exp(self.z_sigma(z_h))+ 1e-8
        sigma = torch.clamp(sigma, min=1e-4)
        mu = 0 + (1 - 0) * (torch.tanh(self.z_mu(z_h)) + 1) / 2
        return pi, sigma, mu    

def gaussian_pdf(y, mu, sigma):
    oneDivSqrtTwoPI = 1.0 / np.sqrt(2.0 * np.pi)
    exponent = -0.5 * ((y - mu) / sigma) ** 2
    return oneDivSqrtTwoPI * torch.exp(exponent) / sigma
def mixture_density(y, pi, mu, sigma):
    K = pi.size(1)  
    mixture_pdf = torch.zeros_like(y)  
    for k in range(K):
        gaussian_k = gaussian_pdf(y, mu[:, k], sigma[:, k])
        mixture_pdf += pi[:, k] * gaussian_k
    return mixture_pdf


model = MDN(n_hidden=83, n_gaussians=4,n_hidden_layers=7,bn=0, dout=0, dout_value=0.35831) #base BCC
model.eval()
optimizer = optim.Adam(model.parameters(), lr=0.00007)
PATH = "checkpoint/model-806.pt" 
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
plt.figure(figsize=(8, 6))
nsmp=48049 #Chosen sample
data_x = np.load('xlo_test.npy')[[nsmp],:] 
comp=np.load('xlo_test_comp.npy')[nsmp,:]
data_y = np.load('ylo_test.npy')[[nsmp],:]
data_y[:,1]=data_y[:,1]+data_y[:,3]
T=data_x[:,-1]*(1892-850)+850
data_x_np = torch.tensor(data_x, dtype=torch.float)
pi, sigma, mu = model(data_x_np)

#print('comp=',comp)
#print('T (K)=',T)
#print('data_y[1]=',data_y[:,1])

weighted_average_mu = torch.sum(pi * mu, dim=-1)
weighted_sigma = torch.sqrt(torch.sum(pi * (sigma ** 2 + (mu - weighted_average_mu.unsqueeze(-1)) ** 2), dim=-1))
weighted_average_mu_np = weighted_average_mu.detach().numpy()
weighted_sigma_np = weighted_sigma.detach().numpy()
y_values = torch.linspace(0, 2, 1000).unsqueeze(1)  
mixture_pdf = mixture_density(y_values, pi, mu, sigma)
area = torch.trapz(mixture_pdf.squeeze(), y_values.squeeze()) #This and next lines are for normalization
mixture_pdf = mixture_pdf / area 
plt.plot(y_values.squeeze(), mixture_pdf.squeeze().detach().numpy(),color='k')
plt.axvline(weighted_average_mu_np, color='red', linestyle='--', linewidth=2, label='Mixture mean')
plt.fill_betweenx(
    [0, plt.ylim()[1]],
    weighted_average_mu_np - weighted_sigma_np,
    weighted_average_mu_np + weighted_sigma_np,
    color='red', alpha=0.2, label='±1σ'
)

print('Mu=',weighted_average_mu_np)
print('Sigma=',weighted_sigma_np)
plt.xlabel("Predicted BCC/B2 Fraction",fontsize=20, fontname='Times New Roman')
plt.ylabel("Normalized Probability Density",fontsize=20, fontname='Times New Roman')
plt.xticks(fontsize=20, fontname='Times New Roman')
plt.yticks(fontsize=20, fontname='Times New Roman') 
plt.xlim([0.01,1.01])
plt.ylim([0,20])
plt.legend(fontsize=20)
plt.tight_layout()
#plt.savefig('5a.png', dpi=600)